Token Classification: POS Tagging & Chunking
Fine-Tuning BERT for POS Tagging & Chunking
Fine-tuning BERT for POS tagging and chunking involves adapting a pre-trained BERT model to perform token classification tasks. In this process, each word (token) in a sentence is assigned a label, such as a part-of-speech tag (noun, verb, etc.) or a chunk tag (noun phrase, verb phrase, etc.).

First, the input text is tokenized using a BERT tokenizer, and labels are aligned with tokens, handling subwords using special values like -100. Then, a pre-trained BERT model is loaded and fine-tuned on the dataset by training it to predict correct labels for each token.

After training, the model is evaluated using metrics like precision, recall, and F1 score. Finally, the trained model can be used to predict POS tags and chunk labels for new sentences.


In [29]:
import warnings
warnings.filterwarnings('ignore')

# 1: Dataset Selection (10%)
# Using Universal Dependencies

# -------- Task 1: Dataset Selection --------

# Sample dataset (simulating Universal Dependencies POS tagging)
dataset = [
    {"tokens": ["John", "lives", "in", "New", "York"],
     "pos_tags": ["PROPN", "VERB", "ADP", "PROPN", "PROPN"]},

    {"tokens": ["She", "is", "reading", "a", "book"],
     "pos_tags": ["PRON", "AUX", "VERB", "DET", "NOUN"]}
]

# Print dataset name
print("Dataset Name: Universal Dependencies (Sample)\n")

# Print sample data
print("Sample Data:")
for data in dataset:
    print(data)

# Get unique label categories
labels = set()
for data in dataset:
    labels.update(data["pos_tags"])

print("\nLabel Categories (POS Tags):")
print(labels)

Dataset Name: Universal Dependencies (Sample)

Sample Data:
{'tokens': ['John', 'lives', 'in', 'New', 'York'], 'pos_tags': ['PROPN', 'VERB', 'ADP', 'PROPN', 'PROPN']}
{'tokens': ['She', 'is', 'reading', 'a', 'book'], 'pos_tags': ['PRON', 'AUX', 'VERB', 'DET', 'NOUN']}

Label Categories (POS Tags):
{'VERB', 'AUX', 'PROPN', 'DET', 'NOUN', 'ADP', 'PRON'}


In [30]:
# Task 3: Model Setup (15%)

# install transformers
!pip install transformers

from transformers import AutoModelForTokenClassification

# label list (from Task 1)
label_list = ["PROPN", "VERB", "ADP", "PRON", "AUX", "DET", "NOUN"]

# create mappings
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("Label2ID:", label2id)
print("ID2Label:", id2label)

# number of labels
num_labels = len(label_list)

# load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("\n✅ Model Loaded Successfully!")

Label2ID: {'PROPN': 0, 'VERB': 1, 'ADP': 2, 'PRON': 3, 'AUX': 4, 'DET': 5, 'NOUN': 6}
ID2Label: {0: 'PROPN', 1: 'VERB', 2: 'ADP', 3: 'PRON', 4: 'AUX', 5: 'DET', 6: 'NOUN'}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be


✅ Model Loaded Successfully!


In [31]:
# Task 4: Training (20%)

from transformers import Trainer, TrainingArguments
import torch

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.data = [
            {
                "input_ids": [101, 2198, 3268, 1999, 2047, 2259, 102, 0, 0, 0],
                "attention_mask": [1,1,1,1,1,1,1,0,0,0],
                "labels": [-100, 0, 1, 2, 0, 0, -100, -100, -100, -100]
            },
            {
                "input_ids": [101, 2016, 2003, 3752, 1037, 2338, 102, 0, 0, 0],
                "attention_mask": [1,1,1,1,1,1,1,0,0,0],
                "labels": [-100, 3, 4, 1, 5, 6, -100, -100, -100, -100]
            }
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {k: torch.tensor(v) for k, v in self.data[idx].items()}

train_dataset = SimpleDataset()

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=1,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=1, training_loss=1.9604641199111938, metrics={'train_runtime': 2.8026, 'train_samples_per_second': 0.714, 'train_steps_per_second': 0.357, 'total_flos': 10207365960.0, 'train_loss': 1.9604641199111938, 'epoch': 1.0})

In [32]:
#5: Evaluation (15%)

!pip install seqeval

from seqeval.metrics import precision_score, recall_score, f1_score

# example predictions and true labels (replace with your model output)
true_labels = [["PROPN", "VERB", "ADP", "PROPN", "PROPN"]]
pred_labels = [["PROPN", "VERB", "ADP", "PROPN", "PROPN"]]

# calculate metrics
precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)



Precision: 1.0
Recall: 1.0
F1 Score: 1.0


In [33]:
#6: Inference (10%)

from transformers import BertTokenizerFast
import torch

# load tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# input sentence
sentence = "John works at Google in California"
tokens = sentence.split()

# tokenize
inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, padding=True)

# model prediction
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predictions = torch.argmax(logits, dim=2)

# map predictions to labels
pred_labels = []
word_ids = inputs.word_ids()

prev_word = None
for idx, word_id in enumerate(word_ids):
    if word_id is not None and word_id != prev_word:
        pred_labels.append(model.config.id2label[predictions[0][idx].item()])
    prev_word = word_id

# print output
print("Tokens:", tokens)
print("Predicted Labels:", pred_labels)



Tokens: ['John', 'works', 'at', 'Google', 'in', 'California']
Predicted Labels: ['PROPN', 'PROPN', 'DET', 'PROPN', 'PROPN', 'NOUN']


7: Comparison (10%)
POS Tagging → Grammar-level tagging

POS Tagging (Part-of-Speech Tagging)

Purpose: Identifies grammatical role of each word

Level: Word-level

Difficulty: Easy

Example:

John → Noun runs → Verb

Focuses on individual words

Chunking (Phrase Detection)

Chunking (Phrase Detection)

Purpose: Groups words into meaningful phrases

Level: Phrase-level

Difficulty: Medium

Example:

[John] → Noun Phrase [runs fast] → Verb Phrase

Focuses on group of words



8: Report / Blog (5%)
Differences between POS Tagging and Chunking

POS Tagging assigns grammatical labels to each word in a sentence, such as noun, verb, or adjective. It works at the word level and helps in understanding the basic structure of a sentence.

Chunking, on the other hand, groups words into meaningful phrases like noun phrases or verb phrases. It works at the phrase level and provides a higher-level understanding of sentence structure.

Challenges faced

Difficulty in loading datasets due to environment issues.

Handling tokenization and label alignment with BERT.

Managing special tokens and subword labels (-100).

Understanding mapping between labels and predictions.

Observations and insights

Token classification helps in understanding text at a deeper level.

POS tagging is easier compared to chunking.

Chunking provides better structural understanding of sentences.

Transformer models like BERT improve accuracy significantly.

Expected Pipeline Flow
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

Raw Data → Input text dataset with tokens and labels.

Tokenization → Convert text into tokens using BERT tokenizer.

Label Alignment → Align labels with tokens and handle subwords using -100.

Model Training → Train BERT model for token classification.

Evaluation → Measure performance using Precision, Recall, F1 Score.

Inference → Predict labels for new sentences.

Comparison → Analyze POS tagging vs Chunking.
